In [ ]:
select * from CUSTOMER_SUPPORT_TICKETS_TO_PROCESS;

In [ ]:
CREATE OR REPLACE TABLE TRANSLATED_CUSTOMER_SUPPORT_TICKETS AS (
SELECT
    TRANSCRIPT_TEXT,
    AI_TRANSLATE(TRANSCRIPT_TEXT, '', 'EN') AS TRANSLATED_TRANSCRIPT,
    SUPPORT_TICKET_DATE,
    CUSTOMER_ID,
    APP_VERSION,
    PLATFORM
FROM customer_support_tickets_to_process
ORDER BY SUPPORT_TICKET_DATE
);

SELECT * FROM TRANSLATED_CUSTOMER_SUPPORT_TICKETS;

In [ ]:
WITH BASE_TRANSCRIPTS AS (
    SELECT
        *
    FROM TRANSLATED_CUSTOMER_SUPPORT_TICKETS
)
SELECT AI_AGG(
    TRANSLATED_TRANSCRIPT,
    'You are an expert at recognizing patterns in customer support tickets. You will receive a set of customer support tickets. Your job is to analyze them and come up with 8 common categories that are mentioned in the tickets. Future tickets will be categorized into the topics that you generate.
*NOTE* 
- Each category should be made up of a maximum of 5 words.
- DO NOT respond with any preamble. Only return 8 categories.' -- Note you can adjust the number of categories here
)
FROM BASE_TRANSCRIPTS;

In [ ]:
CREATE OR REPLACE TABLE GET_TICKET_TOPICS_SUBTOPICS AS (
WITH TRANSLATED_TRANSCRIPTS AS (
    SELECT
        *
    FROM TRANSLATED_CUSTOMER_SUPPORT_TICKETS
)
SELECT
TRANSCRIPT_TEXT as ORIGINAL_TRANSCRIPT,
TRANSLATED_TRANSCRIPT,
-- array copied here
AI_CLASSIFY(TRANSCRIPT_TEXT, 
['Account Summary Issues', 'Duo MFA Issues', 'Passkey Errors', 'App Crashes', 'PDF Generation Issues', 'Login Issues', 'Technical Issues', 'Mobile App Issues']):labels AS CATEGORY_VAL,
REGEXP_REPLACE(CATEGORY_VAL, '[^a-zA-Z]', '') as PRIMARY_CATEGORY
FROM TRANSLATED_TRANSCRIPTS
);

-- Check output
SELECT * FROM GET_TICKET_TOPICS_SUBTOPICS;

In [ ]:
CREATE OR REPLACE PROCEDURE EXTRACT_SUBCATEGORIES_BY_PRIMARY_CATEGORY(
    TABLE_NAME STRING,
    TRANSCRIPT_COLUMN STRING DEFAULT 'transcript_text',
    PRIMARY_CATEGORY_COLUMN STRING DEFAULT 'primary_category'
)
RETURNS TABLE (
    PRIMARY_CATEGORY STRING,
    SUBCATEGORY STRING,
    TOTAL_TRANSCRIPTS NUMBER,
    SAMPLE_TRANSCRIPT STRING
)
LANGUAGE SQL
AS
$$
DECLARE
    result_cursor CURSOR FOR
        SELECT 
            primary_category,
            subcategory,
            total_transcripts,
            sample_transcript
        FROM results_table;
    
    sql_statement STRING;
    
BEGIN
    -- Create a temporary table to store results
    CREATE OR REPLACE TEMPORARY TABLE results_table (
        primary_category STRING,
        subcategory STRING,
        total_transcripts NUMBER,
        sample_transcript STRING
    );
    
    -- Build dynamic SQL query with AI_AGG and FLATTEN to create individual rows
    sql_statement := 
        'INSERT INTO results_table ' ||
        'WITH ai_results AS ( ' ||
            'SELECT ' ||
                '"' || PRIMARY_CATEGORY_COLUMN || '" AS primary_category, ' ||
                'AI_AGG("' || TRANSCRIPT_COLUMN || '", ' ||
                '''Based on these customer support tickets, identify and list the main subcategories or specific types of issues within this category. ' ||
                'Provide a comma-separated list of 3-7 specific subcategories that represent the most common themes or issue types. Also include an other category in the list' ||
                'Focus on actionable, specific subcategories rather than generic ones. ' ||
                'For example, for banking: "Payment Processing Issues, Account Access Problems, Interest Rate Inquiries, Fee Disputes". ' ||
                'Return only the comma-separated list without explanation.''' ||
                ') AS subcategories_list, ' ||
                'COUNT(*) AS total_transcripts, ' ||
                'ANY_VALUE("' || TRANSCRIPT_COLUMN || '") AS sample_transcript ' ||
            'FROM "' || TABLE_NAME || '" ' ||
            'WHERE "' || PRIMARY_CATEGORY_COLUMN || '" IS NOT NULL ' ||
            'GROUP BY "' || PRIMARY_CATEGORY_COLUMN || '" ' ||
        '), ' ||
        'flattened AS ( ' ||
            'SELECT ' ||
                'ar.primary_category, ' ||
                'TRIM(f.value::STRING) AS subcategory, ' ||
                'ar.total_transcripts, ' ||
                'ar.sample_transcript ' ||
            'FROM ai_results ar, ' ||
            'LATERAL FLATTEN(input => SPLIT(ar.subcategories_list, '','')) f ' ||
            'WHERE TRIM(f.value::STRING) != '''' ' ||
        ') ' ||
        'SELECT ' ||
            'primary_category, ' ||
            'subcategory, ' ||
            'total_transcripts, ' ||
            'sample_transcript ' ||
        'FROM flattened ' ||
        'ORDER BY total_transcripts DESC, primary_category, subcategory';
    
    -- Execute the dynamic SQL
    EXECUTE IMMEDIATE sql_statement;
    
    -- Return results using cursor
    OPEN result_cursor;
    RETURN TABLE(result_cursor);
    
END;
$$;

-- call sproc to get the list of sub topics
CALL EXTRACT_SUBCATEGORIES_BY_PRIMARY_CATEGORY('GET_TICKET_TOPICS_SUBTOPICS', 'TRANSLATED_TRANSCRIPT', 'PRIMARY_CATEGORY');

CREATE OR REPLACE TABLE SUPPORT_INTERACTION_TOPICS AS
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

SELECT * FROM SUPPORT_INTERACTION_TOPICS;

In [ ]:
CREATE OR REPLACE TABLE PROCESSED_SUPPORT_TICKETS AS
WITH TRANSLATED_TRANSCRIPTS AS (
    SELECT
        *
    FROM TRANSLATED_CUSTOMER_SUPPORT_TICKETS
),
TOPIC_ANALYSIS AS (
  SELECT
    *,

    -- Sentiment Analysis. Here we detect sentiment for a few aspects that we care about, as well as an overall sentiment number between -1 and 1
    AI_SENTIMENT(TRANSLATED_TRANSCRIPT, ['platform', 'app usability', 'bank representative', 'accomplished desired outcome']) AS SENTIMENT_OBJ,
    SENTIMENT_OBJ:categories[0].sentiment::VARCHAR as OVERALL_SENTIMENT,
    SENTIMENT_OBJ:categories[2].sentiment::VARCHAR as APP_USABILITY_SENTIMENT,
    SENTIMENT_OBJ:categories[3].sentiment::VARCHAR as REPRESENTATIVE_SENTIMENT,
    SENTIMENT_OBJ:categories[4].sentiment::VARCHAR as PLATFORM_SENTIMENT,
    SENTIMENT_OBJ:categories[1].sentiment::VARCHAR as DESIRED_OUTCOME_SENTIMENT,
    SNOWFLAKE.CORTEX.SENTIMENT(TRANSLATED_TRANSCRIPT) as OVERALL_SENTIMENT_NUMBER,

    -- Classify into primary category
   AI_CLASSIFY(
      TRANSLATED_TRANSCRIPT,
      (
        -- Create array of possible primary categories
        SELECT
          ARRAY_AGG(DISTINCT PRIMARY_CATEGORY) AS ALL_CATEGORY_ARRAY
        FROM
          SUPPORT_INTERACTION_TOPICS
      ),
      {
        'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript' -- This may not be necessary, shown to demonstrate the option
      }
    ):labels[0]::text AS PRIMARY_CATEGORY -- Parse primary topic and cast as string
  FROM TRANSLATED_TRANSCRIPTS
),
SUBTOPIC_ANALYSIS AS (
    SELECT
        * exclude(SENTIMENT_OBJ),

        -- Classify into secondary category
        AI_CLASSIFY(
            TRANSLATED_TRANSCRIPT,
            (
                -- Create array of possible secondary categories
                SELECT
                    ARRAY_AGG(DISTINCT SUBCATEGORY) AS ALL_SUBCATEGORY_ARRAY
                FROM
                    CUSTOMER_INTERACTION_TOPICS
                WHERE
                    LOWER(PRIMARY_CATEGORY) = LOWER(PRIMARY_CATEGORY) 
            ),
            {
                'task_description': 'Return a classification of the topic of the customer interaction identified in the support ticket. If you\'re not sure, select "other"'
            }
        ):labels[0]::text AS SECONDARY_CATEGORY -- Parse secondary topic and cast as string
    FROM
        TOPIC_ANALYSIS
)
SELECT
  SUPPORT_TICKET_DATE,
  APP_VERSION,
  PLATFORM,
  CUSTOMER_ID,
  TRANSCRIPT_TEXT as ORIGINAL_TICKET,
  TRANSLATED_TRANSCRIPT as TRANSLATED_TICKET,
  OVERALL_SENTIMENT,
  APP_USABILITY_SENTIMENT,
  REPRESENTATIVE_SENTIMENT,
  PLATFORM_SENTIMENT,
  DESIRED_OUTCOME_SENTIMENT,
  OVERALL_SENTIMENT_NUMBER,
  PRIMARY_CATEGORY,
  SECONDARY_CATEGORY  
FROM
  SUBTOPIC_ANALYSIS;

select * from PROCESSED_SUPPORT_TICKETS;

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE SUPPORT_TICKET_SEARCH_SERVICE
  ON TRANSLATED_TICKET
  ATTRIBUTES APP_VERSION, PLATFORM
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  AS (
    SELECT
        *
    FROM PROCESSED_SUPPORT_TICKETS
);

In [ ]:
SELECT PRIMARY_CATEGORY, ROUND(AVG(OVERALL_SENTIMENT_NUMBER),2) as AVG_SENTIMENT_SCORE from PROCESSED_SUPPORT_TICKETS
GROUP BY PRIMARY_CATEGORY
ORDER BY AVG_SENTIMENT_SCORE DESC;

-- Average sentiment by both primary and secondary topic
SELECT PRIMARY_CATEGORY, SECONDARY_CATEGORY, Round(AVG(OVERALL_SENTIMENT_NUMBER),2) as AVG_SENTIMENT_SCORE from PROCESSED_SUPPORT_TICKETS
GROUP BY PRIMARY_CATEGORY, SECONDARY_CATEGORY
ORDER BY AVG_SENTIMENT_SCORE DESC;

In [ ]:
CREATE OR REPLACE STREAM VOICE_OF_CUSTOMER.PUBLIC.SUPPORT_TICKETS_STREAM
ON TABLE VOICE_OF_CUSTOMER.PUBLIC.CUSTOMER_SUPPORT_TICKETS_TO_PROCESS
APPEND_ONLY = TRUE;

CREATE OR REPLACE TABLE VOICE_OF_CUSTOMER.PUBLIC.PROCESS_SUPPORT_TICKETS (
    SOURCE_TRANSCRIPT VARCHAR,
    TRANSLATED_TRANSCRIPT VARCHAR,
    SENTIMENT NUMBER(3,2),
    PRIMARY_CATEGORY VARCHAR,
    SECONDARY_CATEGORY VARCHAR,
    PROCESSING_TIMESTAMP TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Create a stored procedure that takes any new records, performs the translation, sentiment, and topic categorizations, then writes them to the processed interaction table.
CREATE OR REPLACE PROCEDURE VOICE_OF_CUSTOMER.PUBLIC.PROCESS_NEW_TICKETS_SP()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
  -- Use a MERGE statement if you need to handle updates to transcripts
  -- For an append-only stream and target table, INSERT is sufficient.
  INSERT INTO VOICE_OF_CUSTOMER.PUBLIC.PROCESS_SUPPORT_TICKETS (
    SOURCE_TRANSCRIPT,
    TRANSLATED_TRANSCRIPT,
    SENTIMENT,
    PRIMARY_CATEGORY,
    SECONDARY_CATEGORY
  )
 WITH TRANSLATED_TRANSCRIPTS AS (
    SELECT
        TRANSCRIPT_TEXT,
        AI_TRANSLATE(TRANSCRIPT_TEXT, '', 'EN') AS TRANSLATED_TRANSCRIPT
    FROM CUSTOMER_SUPPORT_TICKETS_TO_PROCESS
),
TOPIC_ANALYSIS AS (
  SELECT
    TRANSCRIPT_TEXT,
    TRANSLATED_TRANSCRIPT,
    SNOWFLAKE.CORTEX.SENTIMENT(TRANSLATED_TRANSCRIPT) AS SENTIMENT,
    AI_CLASSIFY(
      TRANSLATED_TRANSCRIPT,
      (
        SELECT
          ARRAY_AGG(DISTINCT PRIMARY_CATEGORY) AS ALL_TOPICS_ARRAY
        FROM
          SUPPORT_INTERACTION_TOPICS
      ),
      {
        'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript' -- This may not be necessary, shown to demonstrate the option
      }
    ):label::text AS PRIMARY_CATEGORY_FIN -- Parse primary topic and cast as string
  FROM TRANSLATED_TRANSCRIPTS
),
SUBTOPIC_ANALYSIS AS (
    SELECT
        TRANSCRIPT,
        TRANSLATED_TRANSCRIPT,
        SENTIMENT,
        PRIMARY_CATEGORY_FIN as PRIMARY_CATEGORY,
        AI_CLASSIFY(
            TRANSLATED_TRANSCRIPT,
            (
                SELECT
                    ARRAY_AGG(DISTINCT SUBCATEGORY) AS ALL_SUBTOPICS_ARRAY
                FROM
                    SUPPORT_INTERACTION_TOPICS
                WHERE
                    LOWER(PRIMARY_CATEGORY_FIN) = LOWER(PRIMARY_CATEGORY) 
            ),
            {
                'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript'
            }
        ):label::text AS SECONDARY_CATEGORY -- Parse secondary topic and cast as string
    FROM
        TOPIC_ANALYSIS
)
SELECT
  *
FROM
  SUBTOPIC_ANALYSIS
  WHERE TRANSLATED_TRANSCRIPT IS NOT NULL; -- Optional: ensure translation was successful

  RETURN 'Successfully processed new transcripts from stream.';
EXCEPTION
  WHEN OTHER THEN
    RETURN 'Error processing transcripts: ' || SQLERRM;
END;
$$;

-- Step 4: Create a task that reads from that stream every 8 hours
-- This task will execute the stored procedure.
CREATE OR REPLACE TASK VOICE_OF_CUSTOMER.PUBLIC.PROCESS_NEW_TICKETS_TASK
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 */8 * * * UTC' -- Runs every 8 hours (at 00:00, 08:00, 16:00 UTC)
  WHEN SYSTEM$STREAM_HAS_DATA('VOICE_OF_CUSTOMER.PUBLIC.SUPPORT_TICKETS_STREAM') -- Only run if the stream has new data
AS
  CALL VOICE_OF_CUSTOMER.PUBLIC.PROCESS_NEW_TICKETS_SP();

In [ ]:
select * from BANK_REVIEWS_TO_PROCESS;

In [ ]:
CREATE OR REPLACE FUNCTION check_language_udf(str_to_check VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.9' -- Or your preferred supported Python version
PACKAGES = ('langdetect')
HANDLER = 'check_language'
AS
$$
from langdetect import detect

def check_language(str_to_check: str) -> str:
    try:
        # Ensure the input is treated as a string
        text = str(str_to_check)
        lang_short = detect(text)
        return lang_short
    except:
        # Return 'unknown' for empty or undetectable strings.
        return 'unknown'
$$;

In [ ]:
WITH BaseTranscripts AS (
  SELECT
    TRANSCRIPT_TEXT,
    voice_of_customer.public.CHECK_LANGUAGE_UDF(TRANSCRIPT_TEXT) AS original_language
  FROM bank_reviews_to_process
  WHERE LENGTH(TRANSCRIPT_TEXT) > 5
)
  SELECT
    TRANSCRIPT_TEXT,
    original_language,
    CASE
      WHEN original_language = 'en' THEN TRANSCRIPT_TEXT
      ELSE snowflake.cortex.translate(TRANSCRIPT_TEXT,original_language,'en')
    END AS translated_transcript
  FROM BaseTranscripts;

In [ ]:
WITH BaseTranscripts AS (
  SELECT
    TRANSCRIPT_TEXT,
    voice_of_customer.public.CHECK_LANGUAGE_UDF(TRANSCRIPT_TEXT) AS original_language
  FROM voice_of_customer.public.bank_reviews_to_process
  WHERE LENGTH(TRANSCRIPT_TEXT) > 5
),
TranslatedTranscripts AS (
  SELECT
    TRANSCRIPT_TEXT,
    original_language,
    CASE
      WHEN original_language = 'en' THEN TRANSCRIPT_TEXT
      ELSE snowflake.cortex.translate(TRANSCRIPT_TEXT,original_language,'en')
    END AS translated_transcript
  FROM BaseTranscripts
)
SELECT AI_AGG(
    translated_transcript,
    'You are an expert at recognizing patterns in customer reviews. You will receive a set of customer reviewss. Your job is to analyze them and come up with 8 common categories that are mentioned in the reviews. Future reviews will be categorized into the topics that you generate.
*NOTE* 
- Each category should be made up of a maximum of 5 words.
- DO NOT respond with any preamble. Only return 8 categories.' -- Note you can adjust the number of categories here
)
FROM TranslatedTranscripts;

In [ ]:
CREATE OR REPLACE TABLE GET_TOPICS_SUBTOPICS AS (
WITH BaseTranscripts AS (
  SELECT
    TRANSCRIPT_TEXT,
    voice_of_customer.public.CHECK_LANGUAGE_UDF(TRANSCRIPT_TEXT) AS original_language
  FROM voice_of_customer.public.bank_reviews_to_process
  WHERE LENGTH(TRANSCRIPT_TEXT) > 5
),
TranslatedTranscripts AS (
  SELECT
    TRANSCRIPT_TEXT,
    original_language,
    CASE
      WHEN original_language = 'en' THEN TRANSCRIPT_TEXT
      ELSE snowflake.cortex.translate(TRANSCRIPT_TEXT,original_language,'en')
    END AS translated_transcript
  FROM BaseTranscripts
)
SELECT
transcript_text as original_transcript,
translated_transcript,
-- array copied here
AI_CLASSIFY(TRANSCRIPT_TEXT, 
['Wait time inside branch',
'Wait time in drive thru',
'Inexperienced or unprofessional teller',
'Inexperienced or unprofessional banker/advisor',
'Poorly maintained facility',
'Problem accessing security deposit box']):labels AS category_val,
REGEXP_REPLACE(category_val, '[^a-zA-Z]', '') as primary_category
FROM translatedtranscripts
);

-- Check output
SELECT * FROM GET_TOPICS_SUBTOPICS; 

In [ ]:
CREATE OR REPLACE PROCEDURE EXTRACT_SUBCATEGORIES_BY_PRIMARY_CATEGORY(
    TABLE_NAME STRING,
    TRANSCRIPT_COLUMN STRING DEFAULT 'transcript_text',
    PRIMARY_CATEGORY_COLUMN STRING DEFAULT 'primary_category'
)
RETURNS TABLE (
    primary_category STRING,
    subcategory STRING,
    total_transcripts NUMBER,
    sample_transcript STRING
)
LANGUAGE SQL
AS
$$
DECLARE
    result_cursor CURSOR FOR
        SELECT 
            primary_category,
            subcategory,
            total_transcripts,
            sample_transcript
        FROM results_table;
    
    sql_statement STRING;
    
BEGIN
    -- Create a temporary table to store results
    CREATE OR REPLACE TEMPORARY TABLE results_table (
        primary_category STRING,
        subcategory STRING,
        total_transcripts NUMBER,
        sample_transcript STRING
    );
    
    -- Build dynamic SQL query with AI_AGG and FLATTEN to create individual rows
    sql_statement := 
        'INSERT INTO results_table ' ||
        'WITH ai_results AS ( ' ||
            'SELECT ' ||
                '"' || PRIMARY_CATEGORY_COLUMN || '" AS primary_category, ' ||
                'AI_AGG("' || TRANSCRIPT_COLUMN || '", ' ||
                '''Based on these customer service transcripts, identify and list the main subcategories or specific types of issues within this category. ' ||
                'Provide a comma-separated list of 3-7 specific subcategories that represent the most common themes or issue types. Also include an other category in the list' ||
                'Focus on actionable, specific subcategories rather than generic ones. ' ||
                'For example, for banking: "Payment Processing Issues, Account Access Problems, Interest Rate Inquiries, Fee Disputes". ' ||
                'Return only the comma-separated list without explanation.''' ||
                ') AS subcategories_list, ' ||
                'COUNT(*) AS total_transcripts, ' ||
                'ANY_VALUE("' || TRANSCRIPT_COLUMN || '") AS sample_transcript ' ||
            'FROM "' || TABLE_NAME || '" ' ||
            'WHERE "' || PRIMARY_CATEGORY_COLUMN || '" IS NOT NULL ' ||
            'GROUP BY "' || PRIMARY_CATEGORY_COLUMN || '" ' ||
        '), ' ||
        'flattened AS ( ' ||
            'SELECT ' ||
                'ar.primary_category, ' ||
                'TRIM(f.value::STRING) AS subcategory, ' ||
                'ar.total_transcripts, ' ||
                'ar.sample_transcript ' ||
            'FROM ai_results ar, ' ||
            'LATERAL FLATTEN(input => SPLIT(ar.subcategories_list, '','')) f ' ||
            'WHERE TRIM(f.value::STRING) != '''' ' ||
        ') ' ||
        'SELECT ' ||
            'primary_category, ' ||
            'subcategory, ' ||
            'total_transcripts, ' ||
            'sample_transcript ' ||
        'FROM flattened ' ||
        'ORDER BY total_transcripts DESC, primary_category, subcategory';
    
    -- Execute the dynamic SQL
    EXECUTE IMMEDIATE sql_statement;
    
    -- Return results using cursor
    OPEN result_cursor;
    RETURN TABLE(result_cursor);
    
END;
$$;

-- call sproc to get the list of sub topics
CALL EXTRACT_SUBCATEGORIES_BY_PRIMARY_CATEGORY('GET_TOPICS_SUBTOPICS', 'TRANSLATED_TRANSCRIPT', 'PRIMARY_CATEGORY');

-- The output will be a table of the primary_topics and the related sub_topic for each primary.
-- If you want you can manually create this table if you know your primary topics and sub topics you want to classify as.
CREATE OR REPLACE TABLE CUSTOMER_INTERACTION_TOPICS AS
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

SELECT * FROM CUSTOMER_INTERACTION_TOPICS;

In [ ]:
CREATE OR REPLACE TABLE PROCESSED_BANK_REVIEWS AS
WITH BaseTranscripts AS (
  SELECT
    *,
    voice_of_customer.public.CHECK_LANGUAGE_UDF(transcript_text) AS original_language
  FROM voice_of_customer.public.bank_reviews_to_process
  WHERE LENGTH(TRANSCRIPT_TEXT) > 5
),
TranslatedTranscripts AS (
  SELECT
    *,
    CASE
      WHEN original_language = 'en' THEN transcript_text
      ELSE snowflake.cortex.translate(TRANSCRIPT_TEXT,'','en') -- Empty string will detect language
    END AS translated_transcript
  FROM BaseTranscripts
),
TopicAnalysis AS (
  SELECT
    *,

    -- Sentiment Analysis. Here we detect sentiment for a few aspects that we care about, as well as an overall sentiment number between -1 and 1
    AI_SENTIMENT(translated_transcript, ['wait time','cleanliness','bank representative', 'accomplished desired outcome']) AS sentiment_obj,
    sentiment_obj:categories[0].sentiment::VARCHAR as overall_sentiment,
    sentiment_obj:categories[1].sentiment::VARCHAR as cleanliness_sentiment,
    sentiment_obj:categories[2].sentiment::VARCHAR as representative_sentiment,
    sentiment_obj:categories[3].sentiment::VARCHAR as wait_time_sentiment,
    sentiment_obj:categories[3].sentiment::VARCHAR as desired_outcome_sentiment,
    snowflake.cortex.sentiment(translated_transcript) as overall_sentiment_number,

    -- Classify into primary category
   AI_CLASSIFY(
      translated_transcript,
      (
        -- Create array of possible primary categories
        SELECT
          ARRAY_AGG(DISTINCT primary_category) AS all_category_array
        FROM
          CUSTOMER_INTERACTION_TOPICS
      ),
      {
        'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript' -- This may not be necessary, shown to demonstrate the option
      }
    ):labels[0]::text AS primary_category -- Parse primary topic and cast as string
  FROM TranslatedTranscripts
),
SubtopicAnalysis AS (
    SELECT
        * exclude(sentiment_obj),

        -- Classify into secondary category
        AI_CLASSIFY(
            translated_transcript,
            (
                -- Create array of possible secondary categories
                SELECT
                    ARRAY_AGG(DISTINCT subcategory) AS all_subcategory_array
                FROM
                    CUSTOMER_INTERACTION_TOPICS s
                WHERE
                    LOWER(primary_category) = LOWER(primary_category) 
            ),
            {
                'task_description': 'Return a classification of the topic of the customer interaction identified in the review. If you\'re not sure, select "other"'
            }
        ):labels[0]::text AS secondary_category -- Parse secondary topic and cast as string
    FROM
        TopicAnalysis
)
SELECT
  review_date,
  branch_number,
  region_number,
  customer_id,
  transcript_text as original_review,
  original_language,
  translated_transcript as translated_review,
  overall_sentiment,
  cleanliness_sentiment,
  representative_sentiment,
  wait_time_sentiment,
  desired_outcome_sentiment,
  overall_sentiment_number,
  primary_category,
  secondary_category  
FROM
  SubtopicAnalysis;
  
SELECT * FROM PROCESSED_BANK_REVIEWS;

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE review_search_service
  ON translated_review
  ATTRIBUTES region_number, branch_number
  WAREHOUSE = compute_wh
  TARGET_LAG = '1 day'
  AS (
    SELECT
        *
    FROM PROCESSED_BANK_REVIEWS
);

In [ ]:
-- Average sentiment by primary topic
SELECT ROUND(AVG(sentiment),2) as AVG_SENTIMENT_SCORE, primary_category from processed_customer_interactions
GROUP BY primary_category
ORDER BY AVG_SENTIMENT_SCORE DESC;

-- Average sentiment by both primary and secondary topic
SELECT Round(AVG(sentiment),2) as AVG_SENTIMENT_SCORE, category, secondary_category from processed_customer_interactions
GROUP BY primary_category, secondary_category
ORDER BY AVG_SENTIMENT_SCORE DESC;

In [ ]:
CREATE OR REPLACE STREAM voice_of_customer.public.CALL_TRANSCRIPTS_STREAM
ON TABLE voice_of_customer.public.CALL_TRANSCRIPTS
APPEND_ONLY = TRUE; -- Set to TRUE if you only care about new inserts.
                     -- Set to FALSE if you also need to track updates/deletes,
                     -- which would require more complex logic in the SP (e.g., MERGE).

-- Create a table where the processed interactions will be written
CREATE OR REPLACE TABLE voice_of_customer.public.PROCESSED_CALL_TRANSCRIPTS (
    SOURCE_TRANSCRIPT VARCHAR, -- Original transcript from the stream
    ORIGINAL_LANGUAGE VARCHAR,
    TRANSLATED_TRANSCRIPT VARCHAR,
    SENTIMENT NUMBER(3,2), -- Assuming sentiment is a score, adjust precision/scale as needed
    PRIMARY_CATEGORY VARCHAR,
    SECONDARY_CATEGORY VARCHAR,
    PROCESSING_TIMESTAMP TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP() -- Timestamp of when this record was processed
);

-- Create a stored procedure that takes any new records, performs the translation, sentiment, and topic categorizations, then writes them to the processed interaction table.
CREATE OR REPLACE PROCEDURE voice_of_customer.public.PROCESS_NEW_TRANSCRIPTS_SP()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
  -- Use a MERGE statement if you need to handle updates to transcripts
  -- For an append-only stream and target table, INSERT is sufficient.
  INSERT INTO voice_of_customer.public.PROCESSED_CALL_TRANSCRIPTS (
    SOURCE_TRANSCRIPT,
    ORIGINAL_LANGUAGE,
    TRANSLATED_TRANSCRIPT,
    SENTIMENT,
    PRIMARY_CATEGORY,
    SECONDARY_CATEGORY
    -- PROCESSING_TIMESTAMP will use its default value
  )
 WITH BaseTranscripts AS (
  SELECT
    TRANSCRIPT,
    voice_of_customer.DEMO.CHECK_LANGUAGE_UDF(TRANSCRIPT) AS original_language
  FROM voice_of_customer.DEMO.CALL_TRANSCRIPTS
  WHERE LENGTH(TRANSCRIPT) > 5
),
TranslatedTranscripts AS (
  SELECT
    TRANSCRIPT,
    original_language,
    CASE
      WHEN original_language = 'en' THEN TRANSCRIPT
      ELSE snowflake.cortex.complete( -- Could also use CORTEX.TRANSLATE(). Faster, but consumes more credits
        'mixtral-8x7b',
        [
          {
            'role': 'system',
            'content': 'Translate the transcript into English, maintaining the structure of the conversation.'
          },
          { 'role': 'user', 'content': transcript }
        ],
        {}
      ):choices[0]:messages :: VARCHAR
    END AS translated_transcript
  FROM BaseTranscripts
),
TopicAnalysis AS (
  SELECT
    TRANSCRIPT,
    original_language,
    translated_transcript,
    SNOWFLAKE.CORTEX.SENTIMENT(translated_transcript) AS sentiment,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
      translated_transcript,
      (
        SELECT
          ARRAY_AGG(DISTINCT primary_category) AS all_topics_array
        FROM
          CUSTOMER_INTERACTION_TOPICS
      ),
      {
        'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript' -- This may not be necessary, shown to demonstrate the option
      }
    ):label::text AS primary_category_fin -- Parse primary topic and cast as string
  FROM TranslatedTranscripts
),
SubtopicAnalysis AS (
    SELECT
        TRANSCRIPT,
        original_language,
        translated_transcript,
        sentiment,
        primary_category_fin as primary_category,
        SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
            translated_transcript,
            (
                SELECT
                    ARRAY_AGG(DISTINCT subcategory) AS all_subtopics_array
                FROM
                    CUSTOMER_INTERACTION_TOPICS s
                WHERE
                    LOWER(primary_category_fin) = LOWER(primary_category) 
            ),
            {
                'task_description': 'Return a classification of the topic of the customer interaction identified in the transcript'
            }
        ):label::text AS secondary_category -- Parse secondary topic and cast as string
    FROM
        TopicAnalysis
)
SELECT
  *
FROM
  SubtopicAnalysis;
  WHERE translated_transcript IS NOT NULL; -- Optional: ensure translation was successful

  RETURN 'Successfully processed new transcripts from stream.';
EXCEPTION
  WHEN OTHER THEN
    RETURN 'Error processing transcripts: ' || SQLERRM;
END;
$$;

-- Step 4: Create a task that reads from that stream every 8 hours
-- This task will execute the stored procedure.
CREATE OR REPLACE TASK voice_of_customer.public.PROCESS_NEW_TRANSCRIPTS_TASK
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 */8 * * * UTC' -- Runs every 8 hours (at 00:00, 08:00, 16:00 UTC)
  WHEN SYSTEM$STREAM_HAS_DATA('voice_of_customer.public.CALL_TRANSCRIPTS_STREAM') -- Only run if the stream has new data
AS
  CALL voice_of_customer.public.PROCESS_NEW_TRANSCRIPTS_SP();

In [ ]:
create or replace semantic view VOICE_OF_CUSTOMER.PUBLIC.SV_BANK_REVIEWS
	tables (
		CUSTOMER_LIFETIME_VALUE primary key (CUSTOMER_ID),
		PROCESSED_BANK_REVIEWS
	)
	relationships (
		REVIEWSTOLTV as PROCESSED_BANK_REVIEWS(CUSTOMER_ID) references CUSTOMER_LIFETIME_VALUE(CUSTOMER_ID)
	)
	facts (
		CUSTOMER_LIFETIME_VALUE.LIFETIME_VALUE as LIFETIME_VALUE comment='The total amount spent by a customer over their lifetime, representing the customer''s overall value to the business.',
		PROCESSED_BANK_REVIEWS.CLEANLINESS_SENTIMENT as CLEANLINESS_SENTIMENT,
		PROCESSED_BANK_REVIEWS.DESIRED_OUTCOME_SENTIMENT as DESIRED_OUTCOME_SENTIMENT,
		PROCESSED_BANK_REVIEWS.OVERALL_SENTIMENT_NUMBER as overall_sentiment_number comment='a number representing the overall sentiment of the review',
		PROCESSED_BANK_REVIEWS.OVERALL_SENTIMENT_STRING as OVERALL_SENTIMENT comment='A string representing the overall sentiment of the review.',
		PROCESSED_BANK_REVIEWS.REPRESENTATIVE_SENTIMENT as REPRESENTATIVE_SENTIMENT comment='How the customer felt about the representative/bank agent',
		PROCESSED_BANK_REVIEWS.WAIT_TIME_SENTIMENT as WAIT_TIME_SENTIMENT comment='How the customer felt about the wait time.'
	)
	dimensions (
		CUSTOMER_LIFETIME_VALUE.CUSTOMER_ID as CUSTOMER_ID comment='Unique identifier for a customer in the database, used to track and analyze individual customer behavior and value over time.',
		PROCESSED_BANK_REVIEWS.BRANCH_NUMBER as BRANCH_NUMBER with synonyms=('branch_id','location_number','office_number','outlet_number','site_number','store_number') comment='Unique identifier for the bank branch where the review was submitted.',
		PROCESSED_BANK_REVIEWS.CUSTOMER_ID as CUSTOMER_ID with synonyms=('account_holder_id','account_id','client_id','client_number','customer_account_id','customer_number','user_id') comment='Unique identifier for the customer who submitted the review.',
		PROCESSED_BANK_REVIEWS.ORIGINAL_LANGUAGE as ORIGINAL_LANGUAGE,
		PROCESSED_BANK_REVIEWS.ORIGINAL_REVIEW as ORIGINAL_REVIEW,
		PROCESSED_BANK_REVIEWS.PRIMARY_CATEGORY as PRIMARY_CATEGORY comment='Primary category of the review. It is the more general categorization, whereas secondary category is more specific.',
		PROCESSED_BANK_REVIEWS.REGION_NUMBER as REGION_NUMBER with synonyms=('area_code','geographic_area','regional_id','territory_number','zone_number') comment='The region number assigned to the bank branch, which is used to categorize and group branches by geographic location for administrative and reporting purposes.',
		PROCESSED_BANK_REVIEWS.REVIEW_DATE as REVIEW_DATE,
		PROCESSED_BANK_REVIEWS.SECONDARY_CATEGORY as SECONDARY_CATEGORY comment='More specific category than primary category',
		PROCESSED_BANK_REVIEWS.TRANSLATED_REVIEW as TRANSLATED_REVIEW
	)
	comment='This semantic view is for a dataset of reviews for bank branches. It details the branch number, the region number, etc. The facts in this dataset are the sentiment scores for each aspect of the service that we provide. Users can ask for the sentiment of a review, broken down by problem area/category, by branch, by region, etc.'
	with extension (CA='{"tables":[{"name":"CUSTOMER_LIFETIME_VALUE","dimensions":[{"name":"CUSTOMER_ID","sample_values":["159","649","137"]}],"facts":[{"name":"LIFETIME_VALUE","sample_values":["14228","92226","13699"]}]},{"name":"PROCESSED_BANK_REVIEWS","dimensions":[{"name":"BRANCH_NUMBER","sample_values":["1","7","51"]},{"name":"CUSTOMER_ID","sample_values":["649","15","34"]},{"name":"ORIGINAL_LANGUAGE"},{"name":"ORIGINAL_REVIEW"},{"name":"PRIMARY_CATEGORY","sample_values":["Inexperiencedorunprofessionalbankeradvisor","Waittimeindrivethru","Waittimeinsidebranch"]},{"name":"REGION_NUMBER","sample_values":["1","3","6"]},{"name":"SECONDARY_CATEGORY","sample_values":["Account and Service Inquiries","Branch Facilities and Ambiance","Transaction Processing and Speed"]},{"name":"TRANSLATED_REVIEW"}],"facts":[{"name":"CLEANLINESS_SENTIMENT"},{"name":"DESIRED_OUTCOME_SENTIMENT"},{"name":"OVERALL_SENTIMENT_NUMBER","default_aggregation":2,"sample_values":["-0.4656805","0.7266517","0.15460245"]},{"name":"OVERALL_SENTIMENT_STRING","sample_values":["mixed","neutral","negative"]},{"name":"REPRESENTATIVE_SENTIMENT"},{"name":"WAIT_TIME_SENTIMENT"}],"time_dimensions":[{"name":"REVIEW_DATE"}]}],"relationships":[{"name":"REVIEWSTOLTV"}]}');

In [ ]:
create or replace semantic view VOICE_OF_CUSTOMER.PUBLIC.SV_SUPPORT_TICKETS
	tables (
		CUSTOMER_LIFETIME_VALUE primary key (CUSTOMER_ID),
		PROCESSED_SUPPORT_TICKETS
	)
	relationships (
		TICKETS_TO_LTV as PROCESSED_SUPPORT_TICKETS(CUSTOMER_ID) references CUSTOMER_LIFETIME_VALUE(CUSTOMER_ID)
	)
	facts (
		CUSTOMER_LIFETIME_VALUE.CUSTOMER_ID as CUSTOMER_ID comment='Unique identifier for a customer in the database, used to track and analyze individual customer behavior and value over time.',
		CUSTOMER_LIFETIME_VALUE.LIFETIME_VALUE as LIFETIME_VALUE comment='The total amount spent by a customer over their lifetime, representing the customer''s overall value to the business.',
		PROCESSED_SUPPORT_TICKETS.CUSTOMER_ID as CUSTOMER_ID comment='Unique identifier for the customer who submitted the support ticket.',
		PROCESSED_SUPPORT_TICKETS.OVERALL_SENTIMENT_NUMBER as OVERALL_SENTIMENT_NUMBER comment='A numerical representation of the overall sentiment of the support ticket, with negative values indicating a negative sentiment and the magnitude of the value indicating the intensity of the sentiment.'
	)
	dimensions (
		PROCESSED_SUPPORT_TICKETS.APP_USABILITY_SENTIMENT as APP_USABILITY_SENTIMENT comment='The sentiment of the user''s experience with the application, categorized as positive, negative, or mixed, based on the content of the support ticket.',
		PROCESSED_SUPPORT_TICKETS.APP_VERSION as APP_VERSION comment='The version of the application associated with the support ticket.',
		PROCESSED_SUPPORT_TICKETS.DESIRED_OUTCOME_SENTIMENT as DESIRED_OUTCOME_SENTIMENT comment='The sentiment of the desired outcome expressed by the customer in the support ticket, indicating whether the customer''s desired outcome is positive, negative, or unknown.',
		PROCESSED_SUPPORT_TICKETS.ORIGINAL_TICKET as ORIGINAL_TICKET comment='This column contains the original, unedited text of support tickets, including the conversation between the customer and the agent, detailing the issue, troubleshooting steps, and resolution.',
		PROCESSED_SUPPORT_TICKETS.OVERALL_SENTIMENT as OVERALL_SENTIMENT comment='The overall sentiment of the customer''s support ticket, indicating whether the customer''s tone was positive, neutral, or negative.',
		PROCESSED_SUPPORT_TICKETS.PLATFORM as PLATFORM comment='The platform on which the support ticket was submitted, indicating whether the issue occurred on a mobile device, desktop computer, or through a specific application.',
		PROCESSED_SUPPORT_TICKETS.PLATFORM_SENTIMENT as PLATFORM_SENTIMENT comment='The sentiment of the platform used to submit the support ticket, indicating whether the customer''s tone was positive, negative, or unknown.',
		PROCESSED_SUPPORT_TICKETS.PRIMARY_CATEGORY as PRIMARY_CATEGORY comment='The primary category of the support ticket, which represents the main issue or topic that the customer is seeking assistance with, such as Duo MFA issues, account summary issues, or mobile app issues.',
		PROCESSED_SUPPORT_TICKETS.REPRESENTATIVE_SENTIMENT as REPRESENTATIVE_SENTIMENT comment='The sentiment of the representative''s response in the support ticket, indicating whether the tone was neutral, positive, or negative.',
		PROCESSED_SUPPORT_TICKETS.SECONDARY_CATEGORY as SECONDARY_CATEGORY comment='The secondary category of the support ticket, which further classifies the type of issue or inquiry being reported, such as account and service-related questions, errors with transaction processing, or problems with accessing the system or services.',
		PROCESSED_SUPPORT_TICKETS.TRANSLATED_TICKET as TRANSLATED_TICKET comment='This column contains the translated text of customer support tickets, including the conversation between the customer and the agent, detailing the issue, troubleshooting steps, and resolution.',
		PROCESSED_SUPPORT_TICKETS.SUPPORT_TICKET_DATE as SUPPORT_TICKET_DATE comment='Date the support ticket was processed.'
	)
	comment='This semantic view overlooks the support ticket data and connects it to customer lifetime value.'
	with extension (CA='{"tables":[{"name":"CUSTOMER_LIFETIME_VALUE","facts":[{"name":"CUSTOMER_ID","sample_values":["159","649","137"]},{"name":"LIFETIME_VALUE","sample_values":["14228","92226","13699"]}]},{"name":"PROCESSED_SUPPORT_TICKETS","dimensions":[{"name":"APP_USABILITY_SENTIMENT","sample_values":["positive","negative","mixed"]},{"name":"APP_VERSION","sample_values":["1.0.1","1.2.1","1.1.0"]},{"name":"DESIRED_OUTCOME_SENTIMENT","sample_values":["unknown","positive","negative"]},{"name":"ORIGINAL_TICKET","sample_values":["Agent: Thank you for calling Bank Corp support, this is Maya. How can I help today?\\nCustomer: My account summary never fully loads in the app. It''s been like this for three days. This is ridiculous.\\nAgent: I’m very sorry you’re experiencing that. I understand how frustrating it is. Can I confirm the app version and your device?\\nCustomer: Doesn’t matter. It worked before. I just see \\"Loading...\\" forever. I’ve tried restarting and reinstalling.\\nAgent: Thank you. I’ll run through a few steps: please ensure the app is updated, clear the app cache, and check your internet connection. If that fails, try logging in via our website or using mobile data.\\nCustomer: I already did all of that. This is unacceptable. I need access now.\\nAgent: I apologize. I’m escalating this to our technical team as a high-priority issue right now and opening a ticket. Ticket #BC-84291. We’ll aim to respond within 24–48 hours. In the meantime you can access your account on bankcorp.com. We’ll waive any fees caused by this outage if any occur.\\nCustomer: Fine. I expect this fixed fast.","Agent: Thank you for calling Bank Corp support. How can I help today?\\nCustomer: My account page in the mobile app won''t load. I open it and it just spins — this is getting ridiculous.\\nAgent: I’m really sorry you’re dealing with that. Let’s get it fixed. Which device and app version are you using?\\nCustomer: iPhone 12, app says 5.3.1. I’ve restarted the app and my phone already.\\nAgent: Thank you. Can you try force-closing the app, reopening, and checking your Wi‑Fi or cellular connection?\\nCustomer: Did that. Still a blank screen.\\nAgent: Understood. I’ll escalate this to our technical team — there may be a service issue. I’ll log a ticket now and include your device and steps tried. Reference number 842911. We expect an update within 24–48 hours.\\nCustomer: Great. I need access to my account now though.\\nAgent: Meanwhile you can sign in at bankcorp.com to view balances. We’ll send SMS and email updates and can call you when resolved. I apologize for the inconvenience.\\nCustomer: Fine. Thanks.","Agent: Thank you for calling Bank Corp support. This is Maya. How can I help today?\\n\\nCustomer: Hi Maya! I’m getting a passkey error when I try to sign in to the Bank Corp mobile app. Super minor, but I’d love help fixing it.\\n\\nAgent: I’m sorry for the trouble — let’s get you back in. Can you confirm your device model and that biometrics are enabled?\\n\\nCustomer: iPhone 13, biometrics on. Happy to try anything.\\n\\nAgent: Great. Please close the app, reopen, and attempt sign-in. If the passkey error shows, try updating the app from the App Store.\\n\\nCustomer: Okay… update done. Trying now… Nice — still the error.\\n\\nAgent: Thanks. I’ll guide you to re-register the passkey: Settings > Security > Manage Passkeys, remove the old one, then register a new passkey in the app. I’ll stay on the line.\\n\\nCustomer: Done — and yes! I’m in. That was fast, thank you so much.\\n\\nAgent: Wonderful. I’ll log the fix and follow up by email. Anything else I can help with?\\n\\nCustomer: That’s perfect. Really appreciate it!"]},{"name":"OVERALL_SENTIMENT","sample_values":["positive","neutral","negative"]},{"name":"PLATFORM","sample_values":["app","desktop","mobile"]},{"name":"PLATFORM_SENTIMENT","sample_values":["unknown","positive","negative"]},{"name":"PRIMARY_CATEGORY","sample_values":["DuoMFAIssues","AccountSummaryIssues","MobileAppIssues"]},{"name":"REPRESENTATIVE_SENTIMENT","sample_values":["neutral","positive","negative"]},{"name":"SECONDARY_CATEGORY","sample_values":["Account and Service Inquiries","Transaction Processing Errors","Access Issues"]},{"name":"TRANSLATED_TICKET","sample_values":["Agent: Thank you for calling Bank Corp, my name is Ana. How can I help you?\\nCustomer: I am unable to get into my app. Duo MFA is not working every time — “authentication failed” — and the push is not coming. I have restarted the phone, reinstalled the app, nothing. This is unacceptable.\\nAgent: I understand the frustration. Can I confirm your full name and last 4 digits of your social security number?\\nCustomer: João Pereira, social ****1234. I want access now. I need my money and cannot be held up by your failure.\\nAgent: Thank you, João. I am reviewing your profile... I see MFA attempts and error messages. I will try to send a code via SMS and open a technical ticket.\\nCustomer: SMS is not coming either. It seems that your system is down. I demand that you temporarily unblock the Duo requirement or give me an alternative.\\nAgent: I will escalate to level 2 support and request a temporary unblock. You will hear back within 2 hours. Sorry for the inconvenience.","Agent: Thank you for calling Bank Corp support, this is Maya. How can I help today?\\nCustomer: My account summary never fully loads in the app. It''s been like this for three days. This is ridiculous.\\nAgent: I’m very sorry you’re experiencing that. I understand how frustrating it is. Can I confirm the app version and your device?\\nCustomer: Doesn’t matter. It worked before. I just see \\"Loading...\\" forever. I’ve tried restarting and reinstalling.\\nAgent: Thank you. I’ll run through a few steps: please ensure the app is updated, clear the app cache, and check your internet connection. If that fails, try logging in via our website or using mobile data.\\nCustomer: I already did all of that. This is unacceptable. I need access now.\\nAgent: I apologize. I’m escalating this to our technical team as a high-priority issue right now and opening a ticket. Ticket #BC-84291. We’ll aim to respond within 24–48 hours. In the meantime you can access your account on bankcorp.com. We’ll waive any fees caused by this outage if any occur.\\nCustomer: Fine. I expect this fixed fast.","Agent: Hi, this is from Bank Corp, how can I help you?\\nCustomer: Hey! I''m really excited for the app, but when I try to log in it says \\"passkey error.\\" Can I fix it right now?\\nAgent: Sure! Thanks for letting us know. First, confirm you have the latest mobile OS and app?\\nCustomer: yup, all up to date — cool, thanks for keeping the app so cool.\\nAgent: Great. Let''s clear your cache and try to use \\"Log in with backup\\" (SMS) option to get in while we sort this passkey issue. Can you do that?\\nCustomer: Sure, I did that and got in! Thanks, that''s a relief.\\nAgent: Excellent. I''ll be resetting your passkey process in your profile and sending instructions to your email to set up biometrics again.\\nCustomer: cool, I appreciate the quick action. Thanks so much for helping!\\nAgent: You''re welcome! If you need anything else, feel free to call back. Have a great day and enjoy your Bank Corp app."]}],"facts":[{"name":"CUSTOMER_ID","sample_values":["559","352","669"]},{"name":"OVERALL_SENTIMENT_NUMBER","sample_values":["-0.8778791","-0.8295517","-0.70316905"]}],"time_dimensions":[{"name":"SUPPORT_TICKET_DATE","sample_values":["2025-06-14","2025-06-13","2025-06-12"]}]}],"relationships":[{"name":"TICKETS_TO_LTV"}]}');

In [ ]:
create or replace semantic view VOICE_OF_CUSTOMER.PUBLIC.SV_CUSTOMER_LIFETIME_VALUE
	tables (
		CUSTOMER_LIFETIME_VALUE
	)
	facts (
		CUSTOMER_LIFETIME_VALUE.LIFETIME_VALUE as LIFETIME_VALUE comment='The totality of the value a customer is expected to bring to the business over their lifetime, typically calculated by multiplying the average order value by the purchase frequency and customer lifespan.'
	)
	dimensions (
		CUSTOMER_LIFETIME_VALUE.CUSTOMER_ID as CUSTOMER_ID comment='Unique ientifier for each customer in the database, used to track and analyze individual customer behavior and value over time.'
	)
	with extension (CA='{"tables":[{"name":"CUSTOMER_LIFETIME_VALUE","dimensions":[{"name":"CUSTOMER_ID","sample_values":["159","649","137"]}],"facts":[{"name":"LIFETIME_VALUE","sample_values":["14228","92226","13699"]}]}]}');